# Exercise — Module 9: Pipelines & Interpretability (SHAP)
#
**Dataset (โปรดอ่านก่อนใช้งาน):**
- *UCI SECOM* — McCann, M. & Johnston, A. (2008). *SECOM* [Data set]. UCI Machine Learning Repository. https://doi.org/10.24432/C54305 (CC BY 4.0)
- ไฟล์: `datasets/secom.zip` — 1,567 แถว × 590 features, label −1 = PASS / +1 = FAIL, ค่าหายหนัก
#
ทำในโน้ตบุ๊กนี้ได้เลย — รันเซลล์เตรียมข้อมูลก่อน แล้วเติมโค้ดในช่อง `# TODO`

---

**เวลาที่คาดหวังสำหรับแบบฝึกหัดนี้:** ~60 นาที (ทำในห้องช่วง lab — เฉลยเปิดเผยหลังจบช่วง)


In [1]:
from pathlib import Path


def data_dir() -> Path:
    """หาโฟลเดอร์ datasets/ โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "datasets").exists():
            return p / "datasets"
    raise FileNotFoundError("ไม่พบ datasets/ — รันโน้ตบุ๊กจากใน repo")


DATA = data_dir()

## เตรียมข้อมูล (ให้มาแล้ว — รันผ่านก่อนทำข้อ)

ฟอนต์ไทยสำหรับป้ายกราฟ — fallback list รองรับทุกแพลตฟอร์ม (macOS/Windows)

In [2]:
import io  # noqa: E402
import zipfile  # noqa: E402

import matplotlib.pyplot as plt  # noqa: E402
import polars as pl  # noqa: E402

plt.rcParams["font.family"] = [
    "DejaVu Sans",
    "Noto Sans Thai",
    "Leelawadee UI",
    "Tahoma",
    "Thonburi",
]

with zipfile.ZipFile(DATA / "secom.zip") as z:
    feats = pl.read_csv(
        io.BytesIO(z.read("secom.data")),
        separator=" ",
        has_header=False,
        new_columns=[f"feature_{i}" for i in range(1, 591)],
        schema_overrides=[pl.Float64] * 590,
        null_values="NaN",
    )
    lab = pl.read_csv(
        io.BytesIO(z.read("secom_labels.data")),
        separator=" ",
        quote_char='"',
        has_header=False,
        new_columns=["label", "timestamp"],
    )

secom = feats.with_columns(lab["label"].alias("label"))
names = secom.drop("label").columns
X = secom.drop("label").to_numpy()
y = (secom["label"] == 1).to_numpy().astype(int)
n = len(y)
cut = int(n * 0.8)
Xtr, Xte, ytr, yte = X[:cut], X[cut:], y[:cut], y[cut:]
print(f"{n} แถว | train FAIL {ytr.sum()} | test FAIL {yte.sum()}")

1567 แถว | train FAIL 87 | test FAIL 17


## ข้อ A1 — สลับกลยุทธ์ imputation ใน pipeline
#
ใน book ใช้ `strategy="median"` ลองเทียบ 3 กลยุทธ์ — `"mean"`, `"median"`, `SimpleImputer(strategy="constant", fill_value=0.0)` — บน pipeline เดียวกัน (impute → scale → logistic) ด้วย TimeSeriesSplit + average precision แล้วตอบว่าต่างกันมากไหม และเพราะอะไรกับข้อมูลที่ค่าหายกระจายแบบ SECOM

TODO: สร้างฟังก์ชัน make_pipe(imputer) รับ SimpleImputer ที่ยังไม่ fit
คืน Pipeline([("impute", ...), ("scale", ...), ("model", logreg)])
แล้ววน 3 กลยุทธ์ พิมพ์ mean CV AP ของแต่ละแบบ

In [3]:
from sklearn.impute import SimpleImputer  # noqa: E402, F401
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.model_selection import TimeSeriesSplit  # noqa: E402
from sklearn.model_selection import cross_val_score  # noqa: E402, F401
from sklearn.pipeline import Pipeline  # noqa: E402, F401
from sklearn.preprocessing import StandardScaler  # noqa: E402, F401

logreg = LogisticRegression(
    max_iter=5000, class_weight="balanced", random_state=42
)
tscv = TimeSeriesSplit(n_splits=5)

**คำใบ้:** ส่ง estimator ที่ยังไม่ fit เข้า pipeline เสมอ — ห้าม fit เองก่อน (pipeline จะ fit ให้เองต่อ fold) คาดหวังตัวเลขราว 0.07–0.11 ทั้งสามแบบ — ต่างกันไม่มาก

## ข้อ A2 — เปลี่ยนจำนวน feature ที่คัด (k) ข้างใน pipeline
#
ใน book ใช้ `SelectKBest(f_classif, k=20)` ได้ CV AP ≈ 0.108 ลอง k = 5, 20, 50, 150 — คัด **ข้างใน pipeline เสมอ** — แล้วทำตาราง ตอบว่า k เท่าไรดีที่สุดบน CV และมีความเสี่ยงอะไรถ้าเลือก k จากผล CV นี้

TODO: วน k ใน [5, 20, 50, 150]
pipeline: impute(median) → SelectKBest(f_classif, k=k) → scale → logreg
เก็บ mean CV AP แล้วพิมพ์เป็นตาราง

In [4]:
from sklearn.feature_selection import SelectKBest  # noqa: E402, F401
from sklearn.feature_selection import f_classif  # noqa: E402, F401

**คำใบ้:** คาดหวังยอดพีคราว k = 20–50 แล้วตกลงที่ 150 — ส่วนคำถามเรื่องความเสี่ยง: การเลือก k โดยดูผล CV ของ test… เกี่ยวข้องกับ leakage หัวข้อ 2 ของ book อย่างไร (คำตอบ: การตัดสินใจบน CV ของ train ยังโอเค แต่ถ้าไปจูนบน test set คือการซื้อขาดข้อมูลอนาคต)

## ข้อ B1 — สลับโมเดลเป็น XGBoost ใน pipeline เดียวกัน
#
เอา XGBoost (อย่าลืม `scale_pos_weight`) ไปแทนขั้นสุดท้าย วัด CV AP + test AP แล้วเทียบกับ logistic (ใน book: XGBoost CV ≈ 0.187, test ≈ 0.079) — ตอบสั้น ๆ ว่าช่องว่าง CV-vs-test บอกอะไรเรื่องความน่าเชื่อถือของ fold ที่ 4

TODO: pipe_xgb = Pipeline([impute(median), XGBClassifier(...)])
อย่าลืม scale_pos_weight = (ytr==0).sum()/(ytr==1).sum(), eval_metric = "logloss"
cross_val_score ด้วย tscv + "average_precision" แล้ว fit เต็มและวัด test AP

In [5]:
from xgboost import XGBClassifier  # noqa: E402, F401
from sklearn.metrics import average_precision_score  # noqa: E402, F401

**คำใบ้:** fold ที่ 4 ของ XGBoost ใน book ได้ AP ≈ 0.577 ราว 6 เท่าของ fold อื่น — fold นั้นบังเอิญมี FAIL หนาแน่นในช่วง test สั้น ๆ — mean ที่ถูกลากขึ้นโดย fold เดียวคือตัวเลขที่ต้องรายงานพร้อม dispersion เสมอ

## ข้อ C1 — SHAP: ตรวจพิสูจน์โมเดลด้วยมือคุณเอง
#
ใช้ pipeline XGBoost จากข้อ B1: คำนวณ SHAP บน test 200 ชิ้นแรก แล้ว (1) พลอต beeswarm max_display=10, (2) หา **ชิ้นที่โมเดลให้ P(FAIL) สูงสุด** พลอต waterfall, (3) พิมพ์ชื่อ feature 3 อันดับแรกจากค่า |SHAP| เฉลี่ย — แล้วตอบ: ถ้าผู้ตรวจถาม "ทำไมชิ้นนี้ถูกปัด FAIL" คุณจะตอบหนึ่งประโยคด้วยข้อมูลจาก waterfall อย่างไร

TODO: Zte = pd.DataFrame(imputer.transform(Xte), columns=names)
sv = shap.TreeExplainer(model)(Zte.iloc[:200])
beeswarm → หา i_fail = argmax P(FAIL) → waterfall ของ sv[i_fail]
top3 = np.abs(sv.values).mean(axis=0).argsort()[::-1][:3]

In [6]:
import numpy as np  # noqa: E402, F401

import pandas as pd  # noqa: E402, F401
import shap  # noqa: E402, F401

/Users/utarn/projects/ml-metrology/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**คำใบ้:** ดึง imputer/model จาก pipeline ด้วย `pipe.named_steps["impute"]` / `["model"]` — ห้าม fit imputer ใหม่เอง ส่วนประโยคตอบผู้ตรวจ: เอาแท่งใหญ่สุดบน waterfall มาแปลงเป็นภาษาคน เช่น "ชิ้นนี้ถูกปัด FAIL เพราะ feature_060 วัดได้สูงกว่าช่วงปกติของสายผลิตมาก"

## ข้อ D1 — โจทย์คิด (ตอบใน markdown)
#
ทีม QA เสนอว่า: "เรามี 590 features แต่จะทำ SHAP ให้ผู้ตรวจดูแค่ 20 ตัว — งั้นคัด 20 ตัวด้วย SelectKBest กับ **ข้อมูลทั้งหมดก่อน** แล้วค่อยเทรน+วัดผล จะได้ SHAP สวย ๆ 20 ตัว" ชี้ปัญหาของข้อเสนอนี้ 2 ชั้น: ชั้น leakage (จาก book หัวข้อ 2) และชั้นความหมายของ SHAP เอง (ถ้าคัด feature ผิดวิธี SHAP จะอธิบายโมเดลที่ถูกสร้างบนข้อมูลปลอม)

คำตอบข้อ D1: